# B1 — Testes Estatísticos de Drift (Colab)

Adaptador fino que monta o Google Drive, atualiza o repositório e chama `scripts/drift/run_b1.py` em loop sobre os 6 cruzamentos (granularidade × escopo). **Nenhuma lógica científica vive neste notebook** (CLAUDE.md §5).

Aplica KS, CVM, KTS (`MMDDrift`) e LSDD sobre o cache de embeddings BERTimbau `[CLS]`, em janelas mensais e bi-semanais, separados por escopo (global, `mercado`, `nao_mercado`). Para cada par de janelas consecutivas: 1 condição *time-ordered* + 5 *randomized*. Resultado por combo: ~32 pares × 4 testes × 6 condições = ~768 linhas por execução mensal.

**Pré-requisitos:**
- Embeddings já gerados por `compute_embeddings_drift.ipynb` em `MyDrive/ptbr-market-classification/artifacts/drift/embeddings/bertimbau_base_cls/`.
- `data/processado/corpus_opcao7.parquet` no Drive (para `y_original`).
- Runtime → Change runtime type → **GPU (T4 ou L4)** — KTS e LSDD dominam o custo via permutation test com kernel matrix O(N²·d).

**Saída** em `MyDrive/ptbr-market-classification/artifacts/drift/b1_statistical/`:
- Um diretório por combo (`<timestamp>-<granularidade>-<escopo>/`) contendo `metadata.json` + `results.parquet`.

**Tempo esperado** (estimativa grosseira para todo o loop em L4):
- Mensal (32 pares × 6 condições × 3 escopos): ~3–5 h dominado por KTS.
- Bi-semanal (~65 pares × 6 condições × 3 escopos): ~6–10 h.
- **Recomendado**: rodar primeiro só `--granularidade mensal --escopo global` para validar o pipeline antes do loop completo.

## 1. Parâmetros (editar conforme necessário)

In [ ]:
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification.git'  # substituir pela URL do seu fork
RAMO = 'main'

DIR_REPO = '/content/ptbr-market-classification'
DIR_DRIVE = '/content/drive/MyDrive/ptbr-market-classification'

CAMINHO_EMBEDDINGS = f'{DIR_DRIVE}/artifacts/drift/embeddings/bertimbau_base_cls/embeddings.parquet'
CAMINHO_CORPUS = f'{DIR_DRIVE}/data/processado/corpus_opcao7.parquet'
DIR_ARTEFATOS_DRIFT = f'{DIR_DRIVE}/artifacts/drift'

# Combos a rodar. Cada item é (granularidade, escopo). Comente o que
# não quiser executar nesta sessão; o loop é serial e cada combo gera
# diretório próprio em b1_statistical/.
COMBOS = [
    ('mensal',    'global'),
    ('mensal',    'mercado'),
    ('mensal',    'nao_mercado'),
    ('bisemanal', 'global'),
    ('bisemanal', 'mercado'),
    ('bisemanal', 'nao_mercado'),
]

# Smoke test: limita a N pares por execução. Use 2 ou 3 para validar
# pipeline em ~5 min antes do run completo. None = sem limite.
LIMITE_PARES = None

# Permutações para KTS/LSDD. 100 = default Wanderley. Reduzir para 50
# acelera ~2× ao custo de p-values menos precisos.
N_PERMUTATIONS = 100

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar / atualizar repositório

In [ ]:
import os, subprocess

if os.path.exists(DIR_REPO):
    subprocess.run(['git', '-C', DIR_REPO, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'checkout', RAMO], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', RAMO, REPO_URL, DIR_REPO], check=True)

os.chdir(DIR_REPO)
print('cwd =', os.getcwd())

## 4. Instalar dependências

In [ ]:
!pip install -q -r requirements.txt

## 5. Validar GPU + inputs

In [ ]:
import torch
from pathlib import Path

assert torch.cuda.is_available(), (
    'GPU não detectada. Vá em Runtime → Change runtime type e escolha T4 ou L4.'
)
print(f'GPU: {torch.cuda.get_device_name(0)} '
      f'({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB VRAM)')

for caminho, nome in [(CAMINHO_EMBEDDINGS, 'embeddings'), (CAMINHO_CORPUS, 'corpus')]:
    p = Path(caminho)
    assert p.exists(), f'{nome} não encontrado em {p}'
    print(f'{nome}: {p} ({p.stat().st_size / 1024**2:.1f} MB)')

## 6. Loop sobre combos

Cada combo gera um diretório próprio em `artifacts/drift/b1_statistical/`. Falhas em um combo não interrompem os seguintes (try/except por combo). O log da rodada inclui o tempo decorrido.

In [ ]:
import os, subprocess, time

os.environ['PTBR_MC_DIR_DRIFT'] = DIR_ARTEFATOS_DRIFT
os.environ['PTBR_MC_EMBEDDINGS_DRIFT'] = CAMINHO_EMBEDDINGS
os.environ['PTBR_MC_CORPUS_DRIFT'] = CAMINHO_CORPUS

falhas = []
for granularidade, escopo in COMBOS:
    print(f'\n{"="*70}\n>>> {granularidade} / {escopo}\n{"="*70}')
    args = [
        'python', 'scripts/drift/run_b1.py',
        '--embeddings', CAMINHO_EMBEDDINGS,
        '--corpus', CAMINHO_CORPUS,
        '--out', DIR_ARTEFATOS_DRIFT,
        '--granularidade', granularidade,
        '--escopo', escopo,
        '--n-permutations', str(N_PERMUTATIONS),
    ]
    if LIMITE_PARES is not None:
        args += ['--limite-pares', str(LIMITE_PARES)]
    t0 = time.perf_counter()
    res = subprocess.run(args, capture_output=False)
    dt = (time.perf_counter() - t0) / 60
    if res.returncode != 0:
        falhas.append((granularidade, escopo, res.returncode))
        print(f'  FALHA (exit={res.returncode}) após {dt:.1f} min')
    else:
        print(f'  OK em {dt:.1f} min')

print(f'\n{"="*70}')
if falhas:
    print('Combos com falha:')
    for g, e, rc in falhas:
        print(f'  {g} / {e} (exit {rc})')
else:
    print('Todos os combos OK.')

## 7. Resumo dos artefatos gerados

In [ ]:
import json
from pathlib import Path
import pandas as pd

raiz_b1 = Path(DIR_ARTEFATOS_DRIFT) / 'b1_statistical'
print(f'Conteúdo de {raiz_b1}:')
for d in sorted(raiz_b1.glob('*')):
    if not d.is_dir():
        continue
    parquet = d / 'results.parquet'
    metadata = d / 'metadata.json'
    if not (parquet.exists() and metadata.exists()):
        print(f'  {d.name}: INCOMPLETO')
        continue
    m = json.loads(metadata.read_text())
    df = pd.read_parquet(parquet, engine='pyarrow')
    pt = df[df['condicao'] == 'time_ordered']['p_value'].mean()
    pr = df[df['condicao'] == 'randomized']['p_value'].mean()
    print(
        f'  {d.name}: {len(df)} linhas | '
        f'p_value médio time-ordered={pt:.3f} vs randomized={pr:.3f} | '
        f'{m["duracao_segundos"] / 60:.1f} min'
    )